In [1]:
import os
import numpy as np
import tifffile
import sqlite3
import argparse
from datetime import datetime
from cellpose import models

import sys
sys.path.insert(0, '/home/boyesh/akoya_pcf/scripts')
sys.argv = ['segmentation.py', '--project', 'prototype']

from utils import is_already_processed
from ingestion import scan_for_files

In [ ]:
conn = sqlite3.connect("/home/hboyes/akoya_pcf/akoya.db")
cursor = conn.cursor

cursor.execute("""create table if not exists cell_features(
    ...""")

cursor.execute("""create table if not exists cell_intensity(
    ...""")

conn.commit()
conn.close()

In [2]:
ISILON_BASE = os.environ.get("AKOYA_ISILON")
DB_PATH = os.environ.get("AKOYA_DB")

In [5]:
conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM segmentation_results")
print(cursor.fetchall())

cursor.execute("SELECT * FROM pipeline_status")
print(cursor.fetchall())

conn.close()

[(10, 1, 193758, None, 2048, 256, 'Passed', None, '2026-03-05 15:43:07.337086'), (11, 2, 1008553, None, 2048, 256, 'Passed', None, '2026-03-06 04:28:33.175260'), (12, 3, 736326, None, 2048, 256, 'Passed', None, '2026-03-06 10:41:10.134860'), (13, 4, 1080309, None, 2048, 256, 'Passed', None, '2026-03-06 21:59:17.788899'), (14, 5, 398991, None, 2048, 256, 'Passed', None, '2026-03-07 05:31:16.798812')]
[(1, 1, 'Not Started', 'Failed', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None), (2, 2, 'Not Started', 'Failed', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None), (3, 3, 'Not Started', 'Complete', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None), (4, 4, 'Not Started', 'Complete', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None), (5, 5, 'Not Started', 'Failed', 'Passed', 'Not Started', 'Not Started', 'Not Started', None, None)]


In [6]:
conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
cursor = conn.cursor()

cursor.execute("""CREATE TABLE IF NOT EXISTS cell_features(
    cell_id integer primary key autoincrement,
    slide_id integer references slides(slide_id) not null,
    label integer not null,
    area real,
    centroid_x real,
    centroid_y real,
    eccentricity real,
    perimeter real,
    solidity real
)""")

cursor.execute("""CREATE TABLE IF NOT EXISTS cell_intensity(
    intensity_id integer primary key autoincrement,
    cell_id integer references cell_features(cell_id) not null,
    channel_index integer references channel_stats(channel_index) not null,
    channel_name text,
    mean_intensity real not null,
    max_intensity real not null
)""")

conn.commit()
conn.close()
print("Tables created")

Tables created


In [3]:
conn = sqlite3.connect("/home/boyesh/akoya_pcf/akoya.db")
cursor = conn.cursor()

cursor.execute("SELECT * FROM pipeline_status")
print(cursor.fetchall())

cursor.execute("SELECT COUNT(*) FROM cell_features")
print(cursor.fetchall())

cursor.execute("SELECT COUNT(*) FROM cell_intensity")
print(cursor.fetchall())



[(1, 1, 'Not Started', 'Failed', 'Passed', 'Passed', 'Not Started', 'Not Started', None, None), (2, 2, 'Not Started', 'Failed', 'Passed', 'Passed', 'Not Started', 'Not Started', None, None), (3, 3, 'Not Started', 'Complete', 'Passed', 'Passed', 'Not Started', 'Not Started', None, None), (4, 4, 'Not Started', 'Complete', 'Passed', 'Passed', 'Not Started', 'Not Started', None, None), (5, 5, 'Not Started', 'Failed', 'Passed', 'Passed', 'Not Started', 'Not Started', None, None)]
[(3064976,)]
[(24519808,)]


In [4]:
cursor.execute("""SELECT s.slide_id, sr.cell_count, COUNT(cf.cell_id) as extracted_cells
    FROM slides s
    JOIN segmentation_results sr ON s.slide_id = sr.slide_id
    JOIN cell_features cf on s.slide_id = cf.slide_id
    GROUP BY s.slide_id""")
print(cursor.fetchall())

[(1, 193758, 174349), (2, 1008553, 899456), (3, 736326, 661899), (4, 1080309, 973297), (5, 398991, 355975)]


In [6]:
cursor.execute("SELECT * FROM cell_features WHERE slide_id = 1 LIMIT 5")
print(cursor.fetchall())

[(1, 1, 1, 1576.0, 4385.670685279188, 25.42766497461929, 0.5210492574694844, 173.7817459305202, 0.869757174392936), (2, 1, 2, 44.0, 8941.045454545454, 1494.4772727272727, 0.9419611252318283, 27.97056274847714, 0.8301886792452831), (3, 1, 3, 26.0, 8906.653846153846, 1549.2692307692307, 0.6021972256327986, 16.242640687119284, 1.0), (4, 1, 4, 45.0, 8822.6, 1552.5111111111112, 0.6950187917026972, 22.48528137423857, 0.9782608695652174), (5, 1, 5, 41.0, 8829.926829268292, 1613.4146341463415, 0.6967411667399502, 22.727922061357855, 0.9318181818181818)]
